# Time Series Stationarity

## What is Stationarity?

A time series is stationary if it does not exhibit any long term trends or obvious seasonality.

It has:

- A constant variance through time
- A constant mean through time
- No Trend or seasonality

## Visualise Data

In [1]:
# Import packages
import plotly.express as px
import pandas as pd
from statsmodels.tsa.stattools import adfuller
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Read in the data
data = pd.read_csv('/content/drive/MyDrive/TimeSeriesForecasting/Day5/AirPassengers.csv')

In [4]:
def plotting(title, data, x, y, x_label, y_label):
    """General function to plot the passenger data."""
    fig = px.line(data, x=data[x], y=data[y], labels={x: x_label, y: y_label})

    fig.update_layout(template="simple_white", font=dict(size=18),
                      title_text=title, width=1000,
                      title_x=0.5, height=600)

    fig.show()

In [5]:
# Plot the airline passenger data
plotting(title='Airline Passengers', data=data, x='Month',
         y='#Passengers', x_label='Date', y_label='Passengers')

Is this time series stationary? No.

There is a clear increasing trend and the variance of fluctuations are also increasing in time.

To make the time series stationary, we need apply transformations to it.

## Differencing

The most common transformation is differencing.

image.png

Where d(t) is the difference at time t between the series at points y(t) and y(t-1).


In [6]:
# Take the difference and plot it
data["Passenger_Diff"] = data["#Passengers"].diff()

plotting(title='Airline Passengers', data=data, x='Month', y='Passenger_Diff',
         x_label='Date', y_label='Passengers<br>Difference Transform')

Is the data now stationary? No.

The mean is now constant and is oscillating about zero. However, we can clearly see the variance is still increasing through time.

## Logarithm Transform

To stabilise the variance, we apply the natural log transform.

In [7]:
# Take the log and plot it
data["Passenger_Log"] = np.log(data["#Passengers"])

plotting(title='Airline Passengers', data=data, x='Month',
         y='Passenger_Log', x_label='Date', y_label='Passenger<br>Log Transform')

The fluctuations are now on a consistent scale, but there is still a trend. Therefore, we now again have to apply the difference transform.

## Logarithm and Differenc Transform

In [8]:
# Take the difference and log and plot it
data["Passenger_Diff_Log"] = data["Passenger_Log"].diff()

plotting(title='Airline Passengers', data=data, x='Month',
         y='Passenger_Diff_Log', x_label='Date', y_label='Passenger<br>Log and Difference')

In [9]:
data

,Month,#Passengers,Passenger_Diff,Passenger_Log,Passenger_Diff_Log
0,1949-01,112,NaN,4.718499,NaN
1,1949-02,118,6.0,4.770685,0.052186
2,1949-03,132,14.0,4.882802,0.112117
3,1949-04,129,-3.0,4.859812,-0.022990
4,1949-05,121,-8.0,4.795791,-0.064022
...,...,...,...,...,...
139,1960-08,606,-16.0,6.406880,-0.026060
140,1960-09,508,-98.0,6.230481,-0.176399
141,1960-10,461,-47.0,6.133398,-0.097083
142,1960-11,390,-71.0,5.966147,-0.167251


In [10]:
data.dropna(inplace=True)

## Stationarity Test

**ADF Test of Stationarity**

The ADF test (Augmented Dickey-Fuller test) is a statistical test used to check if a time series is stationary—that is, whether its properties like mean and variance are constant over time.

Null Hypothesis: Data is not stationary

If p-value < 0.05:

- We reject the null hypothesis and conclude that the data is stationary.

If p-value > 0.05:
- We fail to reject the null hypothesis and conclude that the data is not stationary.

In [11]:
# Function to run the ADF test
def adf_test(series):
    result = adfuller(series)
    print(f'ADF Statistic: {result[0]}')
    print(f'p-value: {result[1]}')
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'\t{key}: {value}')

    if result[1] <= 0.05:
        print("\nResult: The series is stationary.")
    else:
        print("\nResult: The series is NOT stationary.")

# Run the test on the original data

adf_test(data['Passenger_Diff_Log'])

# print("\n--- ADF Test on Stationary Data ---")
# # Run the test on the final, differenced data
# adf_test(df['Passengers_log_diff_seasonal'])

ADF Statistic: -2.717130598388114
p-value: 0.07112054815086184
Critical Values:
	1%: -3.4825006939887997
	5%: -2.884397984161377
	10%: -2.578960197753906

Result: The series is NOT stationary.


In [12]:
# Seasonal differencing (12 months)
data['Passengers_log_diff_seasonal'] = data['Passenger_Diff_Log'].diff(12)
data.dropna(inplace=True)



In [13]:
plotting(title='Airline Passengers', data=data, x='Month',
         y='Passengers_log_diff_seasonal', x_label='Date', y_label='Passenger<br>Passengers_log_diff_seasonal')

In [14]:
adf_test(data['Passengers_log_diff_seasonal'])

ADF Statistic: -4.4433249418311425
p-value: 0.00024859123113838495
Critical Values:
	1%: -3.4870216863700767
	5%: -2.8863625166643136
	10%: -2.580009026141913

Result: The series is stationary.
